## Data Cleaning And Feature Engineering

In [1]:
## Laoding Raw Ddatasets

import pandas as pd
from pathlib import Path

# Load raw datasets
root_dir = Path.cwd().parent
data_dir = root_dir / "data" / "raw"

campaigns = pd.read_csv(data_dir / "campaigns.csv")
customers = pd.read_csv(data_dir / "customers.csv")
sessions = pd.read_csv(data_dir / "website_sessions.csv")
events = pd.read_csv(data_dir / "website_events.csv")
orders = pd.read_csv(data_dir / "orders.csv")
marketing_spend = pd.read_csv(data_dir / "marketing_spend.csv")

print("All datasets loaded successfully.")



All datasets loaded successfully.


## Initial Data Inspection

In [2]:
datasets = {
    "campaigns": campaigns,
    "customers": customers,
    "sessions": sessions,
    "events": events,
    "orders": orders,
    "marketing_spend": marketing_spend
}

In [3]:

for name,df in datasets.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns ")

campaigns: 19 rows, 5 columns 
customers: 3302 rows, 3 columns 
sessions: 45000 rows, 7 columns 
events: 144097 rows, 5 columns 
orders: 4278 rows, 7 columns 
marketing_spend: 6552 rows, 5 columns 


In [4]:


for name, df in datasets.items(): 
    print(f"\n{'='*50}")
    print(f"  TABLE: {name}")
    print(f"{'='*50}")
    print(f"\n ## Data Types & Missing Values:\n")

    inspection = pd.DataFrame({
        'dtype':df.dtypes,
        'Missing_Count': df.isnull().sum(), 
        'Missing_Pct' : (df.isnull().sum()/len(df)*100).round(2),
        'Duplicate rows' : (df.duplicated().sum())
    })

    print(inspection)




  TABLE: campaigns

 ## Data Types & Missing Values:

                        dtype  Missing_Count  Missing_Pct  Duplicate rows
campaign_id               str              0          0.0               0
campaign_name             str              0          0.0               0
channel                   str              0          0.0               0
campaign_type             str              0          0.0               0
cost_tracking_available  bool              0          0.0               0

  TABLE: customers

 ## Data Types & Missing Values:

                dtype  Missing_Count  Missing_Pct  Duplicate rows
customer_id       str              0          0.0               0
visitor_id        str              0          0.0               0
customer_region   str              0          0.0               0

  TABLE: sessions

 ## Data Types & Missing Values:

                 dtype  Missing_Count  Missing_Pct  Duplicate rows
session_id         str              0          0.0           

### Initial Findings

- no problematic missing values across all 6 tables
- the 96% missing value percentage for order_id is nothing to be concerned about as order_id is recorded only for event_type "purchase"
- numeric columns have proper int64, float64 datatypes
- dates and timestamps are stored as str they need to be fixed. dates columns need to be derived from ts.

### Fixing Dates And Timestamps

In [5]:
sessions['session_start_ts'] = pd.to_datetime(sessions['session_start_ts'])
sessions['session_end_ts'] = pd.to_datetime(sessions['session_end_ts'])
events['event_ts'] = pd.to_datetime(events['event_ts'])
orders['order_ts'] = pd.to_datetime(orders['order_ts'])
marketing_spend['spend_date'] = pd.to_datetime(marketing_spend['spend_date'])


print("Date Columns Fixed")
print("=" * 40)
print(f"session_start_ts  : {sessions['session_start_ts'].dtype}")
print(f"session_end_ts  : {sessions['session_end_ts'].dtype}")
print(f"event_ts          : {events['event_ts'].dtype}")
print(f"order_ts          : {orders['order_ts'].dtype}")
print(f"spend_date        : {marketing_spend['spend_date'].dtype}")

Date Columns Fixed
session_start_ts  : datetime64[us]
session_end_ts  : datetime64[us]
event_ts          : datetime64[us]
order_ts          : datetime64[us]
spend_date        : datetime64[us]


#### Creating date features 

In [6]:
# Session date features
sessions['session_date'] = sessions['session_start_ts'].dt.date
sessions['session_month'] = sessions['session_start_ts'].dt.to_period('M')

# Event date features
events['event_date'] = events['event_ts'].dt.date

# Order date features
orders['order_date'] = orders['order_ts'].dt.date
orders['order_month'] = orders['order_ts'].dt.to_period('M')

# Marketing spend date features
marketing_spend['spend_month'] = marketing_spend['spend_date'].dt.to_period('M')

#### Customer level Features

In [7]:
# Customer summary

customer_summary = orders.groupby('customer_id').agg(
    first_purchase_date=('order_ts', 'min'),
    total_orders=('order_id', 'count'),
    total_revenue=('gross_revenue', 'sum')
).reset_index()

customer_summary['repeat_customer'] = (
    customer_summary['total_orders'] > 1
)

customer_summary.head()

,customer_id,first_purchase_date,total_orders,total_revenue,repeat_customer
0,CUST00001,2025-01-01 11:09:41,1,115.68,False
1,CUST00002,2025-01-01 11:52:20,1,165.02,False
2,CUST00003,2025-01-01 15:12:56,1,208.00,False
3,CUST00004,2025-01-01 16:10:41,2,216.85,True
4,CUST00005,2025-01-01 16:35:56,1,200.27,False


## Order level Features

In [8]:
# Order-level features

orders['net_revenue'] = (
    orders['gross_revenue']
    - orders['discount_amount']
    - orders['refund_amount']
)

orders['discount_pct'] = (
    orders['discount_amount']
    / orders['gross_revenue'] * 100
).round(2)

orders.head()

,order_id,customer_id,session_id,order_ts,gross_revenue,discount_amount,refund_amount,order_date,order_month,net_revenue,discount_pct
0,ORD000001,CUST00001,SES011737,2025-01-01 11:09:41,115.68,0.00,0.0,2025-01-01,2025-01,115.68,0.00
1,ORD000002,CUST00002,SES028264,2025-01-01 11:52:20,165.02,0.00,0.0,2025-01-01,2025-01,165.02,0.00
2,ORD000003,CUST00003,SES009906,2025-01-01 15:12:56,208.00,42.85,0.0,2025-01-01,2025-01,165.15,20.60
3,ORD000004,CUST00004,SES037154,2025-01-01 16:10:41,115.17,0.00,0.0,2025-01-01,2025-01,115.17,0.00
4,ORD000005,CUST00005,SES020451,2025-01-01 16:35:56,200.27,27.06,0.0,2025-01-01,2025-01,173.21,13.51


#### Session-level features



In [9]:
# Session-level features

purchase_sessions = orders['session_id'].unique()

sessions['converted'] = sessions['session_id'].isin(purchase_sessions)

sessions.head()

,session_id,visitor_id,session_start_ts,session_end_ts,campaign_id,device_type,landing_page,session_date,session_month,converted
0,SES000001,VIS00001,2025-09-27 08:56:00,2025-09-27 09:55:00,CMP016,Mobile,Campaign,2025-09-27,2025-09,True
1,SES000002,VIS00002,2025-12-31 19:15:12,2025-12-31 19:50:12,CMP019,Mobile,Product,2025-12-31,2025-12,False
2,SES000003,VIS00003,2026-05-11 19:58:18,2026-05-11 20:29:18,CMP006,Mobile,Campaign,2026-05-11,2026-05,False
3,SES000004,VIS00004,2025-05-07 12:36:24,2025-05-07 13:22:24,CMP009,Mobile,Campaign,2025-05-07,2025-05,False
4,SES000005,VIS00005,2025-05-03 15:58:21,2025-05-03 16:29:21,CMP015,Desktop,Product,2025-05-03,2025-05,True


#### Funnel Features 

In [10]:
type_of_events = ['product_view', 'add_to_cart', 'checkout_started', 'purchase']

for event_type in type_of_events:
    filt = events['event_type'] == event_type
    session_events = events.loc[filt, 'session_id'].unique()

    sessions[event_type] = sessions['session_id'].isin(session_events)

sessions.head()

,session_id,visitor_id,session_start_ts,session_end_ts,campaign_id,device_type,landing_page,session_date,session_month,converted,product_view,add_to_cart,checkout_started,purchase
0,SES000001,VIS00001,2025-09-27 08:56:00,2025-09-27 09:55:00,CMP016,Mobile,Campaign,2025-09-27,2025-09,True,True,True,True,True
1,SES000002,VIS00002,2025-12-31 19:15:12,2025-12-31 19:50:12,CMP019,Mobile,Product,2025-12-31,2025-12,False,True,False,False,False
2,SES000003,VIS00003,2026-05-11 19:58:18,2026-05-11 20:29:18,CMP006,Mobile,Campaign,2026-05-11,2026-05,False,True,False,False,False
3,SES000004,VIS00004,2025-05-07 12:36:24,2025-05-07 13:22:24,CMP009,Mobile,Campaign,2025-05-07,2025-05,False,True,False,False,False
4,SES000005,VIS00005,2025-05-03 15:58:21,2025-05-03 16:29:21,CMP015,Desktop,Product,2025-05-03,2025-05,True,True,True,True,True


In [11]:
# Customer acquisition features
# Find each customer's first recorded session and the campaign associated with it

customer_acquisition = sessions.merge(
    customers,
    on='visitor_id',
    how='left'
)

customer_acquisition = customer_acquisition.sort_values(
    'session_start_ts'
)

customer_acquisition = customer_acquisition.drop_duplicates(
    subset='customer_id',
    keep='first'
)

customer_acquisition = customer_acquisition[
    ['customer_id', 'campaign_id', 'session_start_ts']
]

customer_acquisition.head()

,customer_id,campaign_id,session_start_ts
8178,NaN,CMP010,2025-01-01 08:10:28
4273,CUST01132,CMP008,2025-01-01 08:21:28
25231,CUST01673,CMP006,2025-01-01 09:47:18
15909,CUST00996,CMP002,2025-01-01 10:03:48
3848,CUST01340,CMP015,2025-01-01 10:05:09


In [12]:
#Adding campaign details to identify the acquisition channel

customer_acquisition = customer_acquisition.merge(
    campaigns[['campaign_id', 'campaign_name', 'channel']],
    on='campaign_id',
    how='left'
)

customer_acquisition.head()

,customer_id,campaign_id,session_start_ts,campaign_name,channel
0,NaN,CMP010,2025-01-01 08:10:28,Email Campaign 1,Email
1,CUST01132,CMP008,2025-01-01 08:21:28,Display Campaign 2,Display
2,CUST01673,CMP006,2025-01-01 09:47:18,Paid Social Campaign 3,Paid Social
3,CUST00996,CMP002,2025-01-01 10:03:48,Paid Search Campaign 2,Paid Search
4,CUST01340,CMP015,2025-01-01 10:05:09,Organic Search Campaign 3,Organic Search


In [13]:
# Retention features
# Find the first and second order date for each customer

customer_orders = customers.merge(
    orders[['customer_id', 'order_ts']],
    on='customer_id',
    how='left'
)

customer_orders = customer_orders.sort_values('order_ts')

customer_dates = customer_orders.groupby('customer_id').agg(
    first_order_date=('order_ts', 'min'),
    second_order_date=('order_ts', lambda x: x.iloc[1] if len(x) > 1 else pd.NaT)
).reset_index()

customer_summary = customer_summary.merge(
    customer_dates,
    on='customer_id',
    how='left'
)

customer_summary['days_to_second_order'] = (
    customer_summary['second_order_date']
    - customer_summary['first_order_date']
).dt.days

customer_summary

,customer_id,first_purchase_date,total_orders,total_revenue,repeat_customer,first_order_date,second_order_date,days_to_second_order
0,CUST00001,2025-01-01 11:09:41,1,115.68,False,2025-01-01 11:09:41,NaT,NaN
1,CUST00002,2025-01-01 11:52:20,1,165.02,False,2025-01-01 11:52:20,NaT,NaN
2,CUST00003,2025-01-01 15:12:56,1,208.00,False,2025-01-01 15:12:56,NaT,NaN
3,CUST00004,2025-01-01 16:10:41,2,216.85,True,2025-01-01 16:10:41,2025-01-06 18:32:47,5.0
4,CUST00005,2025-01-01 16:35:56,1,200.27,False,2025-01-01 16:35:56,NaT,NaN
...,...,...,...,...,...,...,...,...
3297,CUST03298,2026-06-30 12:59:54,1,82.95,False,2026-06-30 12:59:54,NaT,NaN
3298,CUST03299,2026-06-30 16:12:28,1,69.14,False,2026-06-30 16:12:28,NaT,NaN
3299,CUST03300,2026-06-30 16:53:02,1,87.53,False,2026-06-30 16:53:02,NaT,NaN
3300,CUST03301,2026-06-30 18:25:11,1,181.18,False,2026-06-30 18:25:11,NaT,NaN


#### Marketing Features

In [14]:
# Marketing features
# Calculate click-through rate and cost per click

marketing_spend['ctr'] = (
    marketing_spend['clicks']
    / marketing_spend['impressions'] * 100
).round(2)

marketing_spend['cpc'] = (
    marketing_spend['spend_amount']
    / marketing_spend['clicks']
).round(2)

marketing_spend.head()

,spend_date,campaign_id,impressions,clicks,spend_amount,spend_month,ctr,cpc
0,2025-01-01,CMP001,2283,60,89.05,2025-01,2.63,1.48
1,2025-01-01,CMP002,2560,128,257.64,2025-01,5.00,2.01
2,2025-01-01,CMP003,3420,199,301.97,2025-01,5.82,1.52
3,2025-01-01,CMP004,1813,30,22.23,2025-01,1.65,0.74
4,2025-01-01,CMP005,4959,110,70.65,2025-01,2.22,0.64


In [15]:
# Add acquisition details to the customer summary

customer_summary = customer_summary.merge(
    customer_acquisition[
        ['customer_id', 'campaign_id', 'campaign_name', 'channel']
    ],
    on='customer_id',
    how='left'
)

In [16]:
customer_summary

,customer_id,first_purchase_date,total_orders,total_revenue,repeat_customer,first_order_date,second_order_date,days_to_second_order,campaign_id,campaign_name,channel
0,CUST00001,2025-01-01 11:09:41,1,115.68,False,2025-01-01 11:09:41,NaT,NaN,CMP011,Email Campaign 2,Email
1,CUST00002,2025-01-01 11:52:20,1,165.02,False,2025-01-01 11:52:20,NaT,NaN,CMP014,Organic Search Campaign 2,Organic Search
2,CUST00003,2025-01-01 15:12:56,1,208.00,False,2025-01-01 15:12:56,NaT,NaN,CMP010,Email Campaign 1,Email
3,CUST00004,2025-01-01 16:10:41,2,216.85,True,2025-01-01 16:10:41,2025-01-06 18:32:47,5.0,CMP014,Organic Search Campaign 2,Organic Search
4,CUST00005,2025-01-01 16:35:56,1,200.27,False,2025-01-01 16:35:56,NaT,NaN,CMP008,Display Campaign 2,Display
...,...,...,...,...,...,...,...,...,...,...,...
3297,CUST03298,2026-06-30 12:59:54,1,82.95,False,2026-06-30 12:59:54,NaT,NaN,CMP006,Paid Social Campaign 3,Paid Social
3298,CUST03299,2026-06-30 16:12:28,1,69.14,False,2026-06-30 16:12:28,NaT,NaN,CMP006,Paid Social Campaign 3,Paid Social
3299,CUST03300,2026-06-30 16:53:02,1,87.53,False,2026-06-30 16:53:02,NaT,NaN,CMP013,Organic Search Campaign 1,Organic Search
3300,CUST03301,2026-06-30 18:25:11,1,181.18,False,2026-06-30 18:25:11,NaT,NaN,CMP004,Paid Social Campaign 1,Paid Social


### Saving the the processed files

In [17]:
# Save processed datasets

processed_dir = root_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

campaigns.to_csv(processed_dir / "campaigns.csv", index=False)
customers.to_csv(processed_dir / "customers.csv", index=False)
sessions.to_csv(processed_dir / "website_sessions.csv", index=False)
events.to_csv(processed_dir / "website_events.csv", index=False)
orders.to_csv(processed_dir / "orders.csv", index=False)
marketing_spend.to_csv(processed_dir / "marketing_spend.csv", index=False)
customer_summary.to_csv(processed_dir / "customer_summary.csv", index=False)

print("All processed datasets saved successfully.")

All processed datasets saved successfully.
